# Notebook 02 — Data Cleaning

## Objectives
In this notebook I take the raw dataset and fix the issues identified in Notebook 01 so that the data is ready for exploration and modelling.

* Fix the `TotalCharges` column — convert from object to float and impute the 11 blank values
* Drop `customerID` which has no predictive value
* Encode the target variable: Churn Yes = 1, No = 0
* Validate the cleaned dataset and save it

## Inputs
* `outputs/datasets/collection/telco_churn_raw.csv`

## Outputs
* `outputs/datasets/collection/telco_churn_cleaned.csv`

---

## 1 — Set Up the Working Directory

In [ ]:
import os

os.chdir('..') if os.path.basename(os.getcwd()) == 'jupyter_notebooks' else None
print('Working directory:', os.getcwd())

## 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## 3 — Load Raw Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_raw.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

## 4 — Fix TotalCharges Column

`TotalCharges` is currently an object. The 11 blank entries belong to customers with zero tenure — they joined recently and have not been billed yet. I replace the blank strings with NaN, convert to float, then fill with the column median.

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('NaN count after conversion:', df['TotalCharges'].isnull().sum())

median_val = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_val)

print(f'Imputed with median: {median_val:.2f}')
print('Remaining NaN values:', df['TotalCharges'].isnull().sum())

## 5 — Drop customerID

`customerID` is a unique identifier with no predictive signal. Keeping it would not help the model and could cause it to overfit noise.

In [ ]:
df = df.drop(columns=['customerID'])
print('customerID dropped. New shape:', df.shape)

## 6 — Encode the Target Variable

The sklearn classifiers I plan to use expect a numeric target, so I map Churn: Yes to 1 and No to 0.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print('Churn value counts after encoding:')
print(df['Churn'].value_counts())

## 7 — Validate the Cleaned Dataset

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())
df.head()

## 8 — Save Cleaned Dataset

In [ ]:
OUT_PATH = 'outputs/datasets/collection/telco_churn_cleaned.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Cleaned dataset saved to: {OUT_PATH}')

---

## Conclusions

* The 11 blank strings in `TotalCharges` were converted to NaN and imputed with the column median.
* `customerID` was dropped — it is a non-predictive unique identifier.
* The `Churn` target was label-encoded: No = 0, Yes = 1.
* The cleaned dataset has **7,043 rows and 20 columns** with no missing values. It is ready for exploratory data analysis.